# arXiv Paper Curator - Week 1: Infrastructure Setup

Build a production-grade RAG system using Docker, PostgreSQL, OpenSearch, FastAPI, Airflow.

## Technology Stack
| Component | Purpose | Port |
|-----------|---------|------|
| **FastAPI** | REST API | 8000 |
| **PostgreSQL** | Paper metadata storage | 5432 |
| **OpenSearch** | Hybrid search engine | 9200/5601 |
| **Apache Airflow** | Workflow automation | 8080 |

## Learning Materials

**Core Technologies:**
- **Docker**: [Tutorial Video](https://www.youtube.com/watch?v=pg19Z8LL06w) | [Docker Compose](https://www.youtube.com/watch?v=SXwC9fSwct8)
- **FastAPI**: [YouTube Series](https://www.youtube.com/playlist?list=PLK8U0kF0E_D6l19LhOGWhVZ3sQ6ujJKq_) | [Documentation](https://fastapi.tiangolo.com/tutorial/)
- **PostgreSQL**: [Beginners Guide](https://www.youtube.com/watch?v=SpfIwlAYaKk) | [FastAPI + PostgreSQL](https://www.youtube.com/watch?v=398DuQbQJq0)
- **OpenSearch**: [Getting Started](https://docs.opensearch.org/latest/getting-started/)
- **Apache Airflow**: [Tutorial Video](https://www.youtube.com/watch?v=Y_vQyMljDsE)

**Development Tools:**
- **VS Code Setup**: [Video Guide](https://www.youtube.com/watch?v=mpk4Q5feWaw)
- **Git Basics**: [Tutorial](https://www.youtube.com/watch?v=zTjRZNkhiEU)
- **UV Package Manager**: [Setup Video](https://www.youtube.com/watch?v=AMdG7IjgSPM)

## Prerequisites

**Required Software:**
- Python 3.12+ ([Download](https://www.python.org/downloads/))
- UV Package Manager ([Install Guide](https://docs.astral.sh/uv/getting-started/installation/))
- Docker Desktop ([Download](https://docs.docker.com/get-docker/))
- Git ([Download](https://git-scm.com/downloads))

**System Requirements:**
- 8GB+ RAM (16GB recommended)
- 20GB+ free disk space

## Setup Instructions

**Before running cells:**
1. Extract/clone project to your system
2. Open terminal in project root (contains `compose.yml`)
3. Run: `uv sync`
4. Start Jupyter: `uv run jupyter notebook`
5. Verify kernel shows project environment (.venv)

In [ ]:
# Environment Check
import sys
from pathlib import Path

python_version = sys.version_info
print(f"Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")
print(f"Environment: {sys.executable}")

if python_version >= (3, 12):
    print("âœ“ Python version compatible")
else:
    print("âœ— Need Python 3.12+")
    exit()

In [ ]:
# Find Project Root
current_dir = Path.cwd()

if current_dir.name == "week1" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = None

if project_root and (project_root / "compose.yml").exists():
    print(f"âœ“ Project root: {project_root}")
else:
    print("âœ— Missing compose.yml - check directory")
    exit()

In [ ]:
# Check Docker
import subprocess

try:
    result = subprocess.run(["docker", "--version"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(f"âœ“ Docker: {result.stdout}")
    else:
        print("âœ— Docker: Not working")
        exit()
except:
    print("âœ— Docker: Not found")
    exit()

In [ ]:
# Check Docker Compose
try:
    result = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(f"âœ“ Docker Compose: {result.stdout.split()[3]}")
    else:
        print("âœ— Docker Compose: Not working")
        exit()
except:
    print("âœ— Docker Compose: Not found")
    exit()

In [ ]:
# Check UV Package Manager
try:
    result = subprocess.run(["uv", "--version"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(f"âœ“ UV: {result.stdout.strip()}")
        print("\nâœ“ All required software ready!")
    else:
        print("âœ— UV: Not working")
        exit()
except:
    print("âœ— UV: Not found")
    exit()

## Start Services

**Command to run (in terminal):**
```bash
cd [project-root]
docker compose up -d
```

**What this does:** Downloads images (first time) and starts all services in background.

In [ ]:
# Check Docker Running
try:
    result = subprocess.run(["docker", "info"], capture_output=True, timeout=5)
    if result.returncode == 0:
        print("âœ“ Docker is running")
    else:
        print("âœ— Docker not running - start Docker Desktop")
        exit()
except:
    print("âœ— Docker daemon not accessible")
    exit()

In [ ]:
# Check Current Containers
import json

try:
    result = subprocess.run(
        ["docker", "compose", "ps", "--format", "json"],
        cwd=str(project_root),
        capture_output=True,
        text=True,
        timeout=10
    )
    
    if result.returncode == 0 and result.stdout.strip():
        print("Current containers:")
        for line in result.stdout.strip().split('\n'):
            if line.strip():
                try:
                    container = json.loads(line)
                    service = container.get('Service', 'unknown')
                    state = container.get('State', 'unknown')
                    print(f"  â€¢ {service}: {state}")
                except:
                    pass
    else:
        print("No containers running")
        
except Exception as e:
    print("Could not check containers")

## Service Health Verification

All services start automatically. Check their health status:

In [ ]:
# Service Health Check
EXPECTED_SERVICES = {
    'api': 'FastAPI REST API server',
    'postgres': 'PostgreSQL database',
    'opensearch': 'OpenSearch search engine', 
    'opensearch-dashboards': 'OpenSearch web dashboard',
    'airflow': 'Workflow automation (optional - may be off)'
}

try:
    result = subprocess.run(
        ["docker", "compose", "ps", "--format", "json"],
        cwd=str(project_root),
        capture_output=True,
        text=True,
        timeout=15
    )
    
    if result.returncode == 0:
        print("SERVICE STATUS")
        print("=" * 70)
        print(f"{'Service':<20} {'State':<15} {'Status':<15} {'Notes'}")
        print("-" * 70)
    else:
        print("Could not get service status")
        exit()
        
except Exception as e:
    print(f"Error checking services: {e}")
    exit()

# Parse Service Status
found_services = set()
service_states = {}

if result.stdout.strip():
    for line in result.stdout.strip().split('\n'):
        if line.strip():
            try:
                container = json.loads(line)
                service = container.get('Service', 'unknown')
                state = container.get('State', 'unknown')
                health = container.get('Health', 'no check')
                
                found_services.add(service)
                service_states[service] = {'state': state, 'health': health}
                
                if state == 'running' and health in ['healthy', 'no check']:
                    indicator = "âœ“"
                    notes = "Ready"
                elif state == 'running' and health == 'unhealthy':
                    indicator = "âš "
                    notes = "Starting up..."
                elif state == 'exited':
                    indicator = "âœ—"
                    notes = "Failed to start"
                else:
                    indicator = "?"
                    notes = f"Status: {state}"
                
                print(f"{indicator} {service:<18} {state:<14} {health:<14} {notes}")
                
            except json.JSONDecodeError:
                pass

In [ ]:
# Check Missing Services
missing_services = set(EXPECTED_SERVICES.keys()) - found_services

if missing_services:
    print("\nMISSING SERVICES:")
    print("-" * 70)
    for service in missing_services:
        description = EXPECTED_SERVICES[service]
        if service == 'airflow':
            print(f"âš  {service:<18} not running    {'(Optional)':<14} {description}")
        else:
            print(f"âœ— {service:<18} not running    {'Required':<14} {description}")

failed_services = [s for s, info in service_states.items() 
                  if info['state'] in ['exited', 'restarting'] or info['health'] == 'unhealthy']

if failed_services:
    print(f"\nTROUBLESHOOTING:")
    for service in failed_services:
        print(f"   docker compose logs {service}")
elif missing_services and 'airflow' not in missing_services:
    print(f"\nACTION NEEDED:")
    print("Start missing services: docker compose up -d")

### 1. FastAPI - REST API Service

**Interactive Exploration:**

You can explore and test the FastAPI service in several ways:
- **API Documentation**: http://localhost:8000/docs (Interactive Swagger UI)
- **Alternative Docs**: http://localhost:8000/redoc (ReDoc interface)
- **Source Code**: Located in `src/routers/` directory

Let's test the API endpoints and explore the documentation:

In [ ]:
# Test FastAPI Health
import requests

try:
    response = requests.get("http://localhost:8000/health", timeout=5)
    if response.status_code == 200:
        data = response.json()
        print("âœ“ FastAPI is responding")
        print(f"Status: {data.get('status', 'unknown')}")
    else:
        print(f"âš  API returned status: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("âœ— API not responding - wait 1-2 minutes")
except Exception as e:
    print(f"âœ— API test error: {e}")

In [ ]:
# PRODUCTION INSIGHTS
print("\n" + "="*60)
print("  PRODUCTION INSIGHT (Online Sessions Only)")
print("="*60)
print("â“ How are they scaled?")
print("â“ What are the bottlenecks?")
print("â“ How are they monitored and managed?")
print("â“ How are they integrated with other systems?")
print("â“ What are the best practices for using these systems?")
print("â“ How are these systems used and deployed in production?")
print("â“ How are they tested? in terms of load and performance?")
print("â†’ Learn these production secrets in our online walkthrough sessions!")
print("="*60)

### 2. Apache Airflow - Workflow Automation

**Interactive Exploration:**

Apache Airflow manages data pipelines and automated workflows. You can explore it through:
- **Web Dashboard**: http://localhost:8080 
- **Login**: Username: `admin`, Password: `admin`
- **Source Code**: Located in `airflow/dags/` directory

**About Our Setup:**
We're using Airflow 2.10.3 with:
- PostgreSQL as the metadata backend (not SQLite)
- LocalExecutor for parallel task execution
- Simple admin credentials for easy access

Let's test Airflow:

In [ ]:
# Airflow Connection Info
print("Airflow Access:")
print("=" * 40)
print("URL: http://localhost:8080")
print("Username: admin")
print("Password: admin")
print("=" * 40)

In [ ]:
# Test Airflow Health
try:
    response = requests.get("http://localhost:8080/health", timeout=5)
    if response.status_code == 200:
        health_data = response.json()
        print("âœ“ Airflow is healthy")
        
        # Show status details
        metadatabase = health_data.get('metadatabase', {})
        scheduler = health_data.get('scheduler', {})
        
        print(f"\nMetadatabase: {metadatabase.get('status', 'unknown')}")
        print(f"Scheduler: {scheduler.get('status', 'unknown')}")
        
        print(f"\nAirflow Login:")
        print(f"URL: http://localhost:8080")
        print(f"Username: admin")
        print(f"Password: admin")
    else:
        print(f"âš  Airflow returned: {response.status_code}")
        
except requests.exceptions.ConnectionError:
    print("âœ— Airflow not responding - wait 2-3 minutes")
except Exception as e:
    print(f"âœ— Airflow test error: {e}")

### 3. OpenSearch - Hybrid database

**Interactive Exploration:**

OpenSearch provides full-text search and analytics capabilities:
- **API Endpoint**: http://localhost:9200 
- **Dashboards UI**: http://localhost:5601 (Web interface)
- **Source Code**: Located in `src/services/opensearch/` directory

**Important for Students:** 
- âœ… Use http://localhost:5601 for web interface
- âœ… Use Dev Tools in Dashboards for API queries

Let's test OpenSearch and explore its capabilities:

In [ ]:
# Test 1: Check OpenSearch Dashboards Web Interface
# This is the proper way for students to interact with OpenSearch

dashboards_url = "http://localhost:5601"

try:
    # Test if Dashboards is accessible
    response = requests.get(f"{dashboards_url}/api/status", timeout=10, allow_redirects=True)
    if response.status_code == 200:
        print("âœ“ OpenSearch Dashboards is accessible!")
        print("âœ“ Web interface is ready for exploration")
        
        print("\n Web Interface Access:")
        print("=" * 40)
        print(f"Main Dashboard: {dashboards_url}")
        print(f"Dev Tools: {dashboards_url}/app/dev_tools")
        print("=" * 40)
        
        print("\n Student Learning Activities:")
        print("1. Explore the Dashboard:")
        print("   â€¢ Visit http://localhost:5601")
        print("   â€¢ Navigate through the interface")
        print("   â€¢ Check out the 'Discover' tab")
        
        print("\n2. Use Dev Tools for API Queries:")
        print("   â€¢ Go to Dev Tools")
        print("   â€¢ Try: GET /_cluster/health")
        print("   â€¢ Try: GET /_cat/indices?v")
        print("   â€¢ Try: GET /_cluster/stats")
        print("   â€¢ Check the learning material for more information")
        
    else:
        print(f"âš  Dashboards returned status: {response.status_code}")
        print("Interface may still be starting up")
        
except requests.exceptions.ConnectionError:
    print("âœ— OpenSearch Dashboards not accessible yet")
    print("Wait 2-3 minutes for full startup")
    
except requests.exceptions.Timeout:
    print("âš  Dashboards request timed out")
    print("This is normal during startup - try again in a few minutes")
    
except Exception as e:
    print(f"âœ— Error accessing Dashboards: {e}")
    print("Check container status: docker compose ps")

In [ ]:
# PRODUCTION DEPLOYMENT INSIGHT
print("\n" + "="*60)
print("ðŸŽ¯ PRODUCTION INSIGHT (Online Sessions Only)")
print("="*60)
print("â“ Why companies use OpenSearch?")
print("â“ What all is achievable with OpenSearch?")
print("â“ How does OpenSearch handle billions of documents?")
print("â“ How do companies search through billions of documents?")
print("â“ How do e-commerce giants search millions of products instantly?")
print("â†’ Learn these production secrets in our online walkthrough sessions!")
print("="*60)

In [ ]:
# PRODUCTION DEPLOYMENT INSIGHT
print("\n" + "="*60)
print("ðŸŽ¯ PRODUCTION INSIGHT (Online Sessions Only)")
print("="*60)
print("â“ What are the real issues with LLMs when in production?")
print("â“ What is the difference between fine-tuned LLM and RAG?")
print("â“ How do companies serve LLMs without burning through cash?")
print("â†’ Learn these production secrets in our online walkthrough sessions!")
print("="*60)

### 5. PostgreSQL - Database Storage

**Interactive Exploration:**

PostgreSQL stores all structured data for our application:
- **Connection**: localhost:5432
- **Database**: rag_db
- **Username/Password**: rag_user / rag_password
- **GUI Tool Recommendation**: DBeaver (free database client)

Let's test the database connection and explore the schema:

In [ ]:
# Test 1: Check PostgreSQL Connection (Basic)
# Let's verify PostgreSQL is accepting connections

def test_postgres_connection():
    """Test PostgreSQL connection using simple socket check."""
    import socket
    
    try:
        # Test if PostgreSQL port is open
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(3)
        result = sock.connect_ex(('localhost', 5432))
        sock.close()
        
        if result == 0:
            print("âœ“ PostgreSQL is accepting connections on port 5432!")
            return True
        else:
            print("âœ— PostgreSQL port is not accessible")
            return False
            
    except Exception as e:
        print(f"âœ— Could not test PostgreSQL: {e}")
        return False

postgres_available = test_postgres_connection()

if postgres_available:
    print("\n  Database Connection Details:")
    print("â€¢ Host: localhost")
    print("â€¢ Port: 5432") 
    print("â€¢ Database: rag_db")
    print("â€¢ Username: rag_user")
    print("â€¢ Password: rag_password")
    
    print("\n  Recommended GUI Tools:")
    print("â€¢ DBeaver (Free): https://dbeaver.io/download/")
    print("â€¢ pgAdmin: https://www.pgadmin.org/download/")

In [ ]:
# Test PostgreSQL Connection
try:
    import psycopg2
    
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        database="rag_db", 
        user="rag_user",
        password="rag_password"
    )
    
    print("âœ“ PostgreSQL connected")
    cursor = conn.cursor()
    
except ImportError:
    print("âš  psycopg2 not installed - basic connection only")
    exit()
except Exception as e:
    print(f"âœ— Database connection failed: {e}")
    exit()

In [ ]:
# Check Database Tables
cursor.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

all_tables = cursor.fetchall()

app_tables = []
airflow_tables = []

for (table_name,) in all_tables:
    if table_name in ['papers', 'users', 'embeddings']:
        app_tables.append(table_name)
    else:
        airflow_tables.append(table_name)

print(f"Found {len(all_tables)} total tables")
print(f"Application tables: {len(app_tables)}")
print(f"Airflow tables: {len(airflow_tables)}")

for table in app_tables:
    print(f"  â€¢ {table}")

if not app_tables:
    print("  No application tables yet (expected in Week 1)")
    
cursor.close()
conn.close()

In [ ]:
# PRODUCTION DEPLOYMENT INSIGHT
print("\n" + "="*60)
print("ðŸŽ¯ PRODUCTION INSIGHT (Online Sessions Only)")
print("="*60)
print("â“ How do companies handle millions of transactions with PostgreSQL?")
print("â“ What's the secret to zero-downtime database migrations?")
print("â†’ Learn these production secrets in our online walkthrough sessions!")
print("="*60)

### Service Health Summary and Next Steps

Based on the interactive tests above:

**If all services show âœ“**: 
- ðŸŽ‰ Congratulations! Your infrastructure is ready
- All services are healthy and responding correctly
- You can explore each service using the links and instructions provided

**If some services show âœ—**:
- Don't worry! Services take time to start
- Wait 2-3 minutes and re-run the test cells
- OpenSearch and Airflow take the longest (up to 5 minutes)

**Service Access Points:**
- **FastAPI Documentation**: http://localhost:8000/docs - Interactive API testing
- **Airflow Dashboard**: http://localhost:8080 (admin/admin) - Workflow management
- **OpenSearch Dashboards**: http://localhost:5601 - Dashboard and user interface + analytics
- **OpenSearch API**: http://localhost:9200 - Direct API access
- **PostgreSQL**: http://localhost:5432 - Use DBeaver or similar tools

**Hands-On Learning Activities:**

1. **FastAPI**: Test endpoints in the interactive documentation
2. **Airflow**: Login and trigger a DAG manually  
3. **OpenSearch**: Try queries in the Dev Tools
4. **PostgreSQL**: Install DBeaver and explore the database structure

**Common Issues:**
- "Connection refused" â†’ Service still starting
- "Port in use" â†’ Another application using the port  
- Container restarting â†’ Check logs with `docker compose logs [service-name]`

## Troubleshooting

**Common Issues:**
- **Connection refused** â†’ Service still starting (wait 2-3 minutes)
- **Port in use** â†’ Stop conflicting application or change ports
- **Container restarting** â†’ Check logs: `docker compose logs [service-name]`
- **Out of memory** â†’ Increase Docker Desktop memory allocation

**Reset everything:** `docker compose down && docker compose up -d`

## Week 1 Complete

**Service Access Points:**
- **API**: http://localhost:8000/docs
- **Airflow**: http://localhost:8080 (admin/admin)  
- **OpenSearch**: http://localhost:5601
- **PostgreSQL**: localhost:5432 (rag_user/rag_password)

**Success Criteria:**
- [ ] All services healthy in status check
- [ ] API documentation accessible
- [ ] Airflow dashboard loads
- [ ] OpenSearch interface works

**Next:** Keep services running or restart with `docker compose up -d`

## Project Commands

**Makefile shortcuts:**
```bash
make start    # Start all services  
make status   # Check service status
make logs     # View logs
make health   # Check service health
make stop     # Stop all services
make help     # View all commands
```

**Next:** Read the main `README.md` for complete project documentation.